In [1]:
from google.colab import files
uploaded = files.upload()

Saving mit-bih-arrhythmia-database-1.0.0 (1).zip to mit-bih-arrhythmia-database-1.0.0 (1).zip


In [3]:
import zipfile

zip_path = "mit-bih-arrhythmia-database-1.0.0 (1).zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/")

In [10]:
# =========================
# INSTALL
# =========================
!pip install wfdb torch scikit-learn pandas matplotlib

# =========================
# IMPORTS
# =========================
import os
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import wfdb
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# =========================
# CONFIG
# =========================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_PATH =  "/content/mit-bih-arrhythmia-database-1.0.0"

RECORDS = ["100","101","102","103","104","105","106","107","108","109"]

CLASS_MAP = {
    "N": "N", "L": "N", "R": "N",
    "A": "S", "a": "S",
    "V": "V", "E": "V",
    "F": "F"
}

WINDOW = 200
NUM_CLIENTS = 3
ROUNDS = 5
LOCAL_EPOCHS = 2
LR = 1e-3
MU = 0.01

# =========================
# LOAD DATA
# =========================
def load_data():
    X, y = [], []

    for rec in RECORDS:
        path = os.path.join(DATA_PATH, rec)
        record = wfdb.rdrecord(path)
        ann = wfdb.rdann(path, 'atr')

        signal = record.p_signal[:,0]

        for i, idx in enumerate(ann.sample):
            sym = ann.symbol[i]

            if sym not in CLASS_MAP:
                continue

            if idx+WINDOW < len(signal):
                seg = signal[idx:idx+WINDOW]
                X.append(seg)
                y.append(CLASS_MAP[sym])

    return np.array(X), np.array(y)

print("Loading data...")
X, y = load_data()

# Normalize
X = (X - np.mean(X)) / (np.std(X) + 1e-8)

# Encode labels
le = LabelEncoder()
y = le.fit_transform(y)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Clients split
client_data = []
split = len(X_train)//NUM_CLIENTS

for i in range(NUM_CLIENTS):
    s = i*split
    e = (i+1)*split
    client_data.append((X_train[s:e], y_train[s:e]))

# =========================
# CNN MODEL
# =========================
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1,16,5,padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16,32,5,padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32,64,3,padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),

            nn.Flatten(),
            nn.Linear(64,num_classes)
        )

    def forward(self,x):
        return self.net(x)

# =========================
# TRAIN CLIENT
# =========================
def train_client(model,data,global_weights=None,prox=False):
    model.train()
    opt = optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss()

    X,y = data
    X = torch.tensor(X,dtype=torch.float32).unsqueeze(1).to(DEVICE)
    y = torch.tensor(y,dtype=torch.long).to(DEVICE)

    for _ in range(LOCAL_EPOCHS):
        opt.zero_grad()
        out = model(X)
        loss = loss_fn(out,y)

        if prox:
            prox_term=0
            for w,gw in zip(model.parameters(),global_weights):
                prox_term += ((w-gw)**2).sum()
            loss += (MU/2)*prox_term

        loss.backward()
        opt.step()

    return model.state_dict()

# =========================
# AGGREGATION
# =========================
def fedavg(ws):
    avg = copy.deepcopy(ws[0])
    for k in avg:
        for i in range(1,len(ws)):
            avg[k]+=ws[i][k]
        avg[k]/=len(ws)
    return avg


m, v = {}, {}
t = 0

def fedadam(global_w, local_ws, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
    global m, v, t
    t += 1

    new_w = copy.deepcopy(global_w)

    for k in global_w:
        # 🔥 IMPORTANT: compute delta
        delta = sum([(global_w[k] - w[k]) for w in local_ws]) / len(local_ws)

        if k not in m:
            m[k] = torch.zeros_like(delta)
            v[k] = torch.zeros_like(delta)

        # Adam updates
        m[k] = beta1 * m[k] + (1 - beta1) * delta
        v[k] = beta2 * v[k] + (1 - beta2) * (delta ** 2)

        m_hat = m[k] / (1 - beta1 ** t)
        v_hat = v[k] / (1 - beta2 ** t)

        new_w[k] = global_w[k] - lr * m_hat / (torch.sqrt(v_hat) + eps)

    return new_w

# =========================
# EVALUATE
# =========================
def evaluate(model):
    model.eval()
    X = torch.tensor(X_test,dtype=torch.float32).unsqueeze(1).to(DEVICE)

    preds = model(X).argmax(1).cpu().numpy()

    acc = accuracy_score(y_test,preds)
    precision = precision_score(y_test,preds,average='weighted',zero_division=0)
    recall = recall_score(y_test,preds,average='weighted',zero_division=0)
    f1 = f1_score(y_test,preds,average='weighted',zero_division=0)
    cm = confusion_matrix(y_test,preds)

    return acc,precision,recall,f1,cm

# =========================
# TRAIN LOOP
# =========================
results = {}

def run(method):
    global_model = CNN(len(np.unique(y))).to(DEVICE)
    global_w = global_model.state_dict()

    print(f"\n=== {method.upper()} ===")

    for r in range(ROUNDS):
        ws=[]

        for data in client_data:
            m = CNN(len(np.unique(y))).to(DEVICE)
            m.load_state_dict(global_w)

            if method=="fedprox":
                w = train_client(m,data,m.parameters(),prox=True)
            else:
                w = train_client(m,data)

            ws.append(w)

        if method=="fedavg":
            global_w = fedavg(ws)
        elif method=="fedadam":
            global_w = fedadam(global_w,ws)
        else:
            global_w = fedavg(ws)

        global_model.load_state_dict(global_w)

        acc,p,r,f1,cm = evaluate(global_model)

        print(f"Round {r+1}: Acc={acc:.4f}, F1={f1:.4f}")

    results[method] = [acc,p,r,f1]

# =========================
# RUN ALL
# =========================
run("fedavg")
run("fedprox")
run("fedadam")

# =========================
# FINAL TABLE
# =========================
df = pd.DataFrame(results, index=["Accuracy","Precision","Recall","F1"]).T
print("\nFINAL RESULTS:\n",df)

df.to_csv("final_results.csv")

Loading data...

=== FEDAVG ===
Round 1.0505344995140913: Acc=0.0505, F1=0.0149
Round 1.2756721736313574: Acc=0.2757, F1=0.4028
Round 1.952057013281503: Acc=0.9521, F1=0.9287
Round 1.952057013281503: Acc=0.9521, F1=0.9287
Round 1.952057013281503: Acc=0.9521, F1=0.9287

=== FEDPROX ===
Round 1.0495626822157433: Acc=0.0496, F1=0.0129
Round 1.7635244574020086: Acc=0.7635, F1=0.8252
Round 1.952057013281503: Acc=0.9521, F1=0.9287
Round 1.952057013281503: Acc=0.9521, F1=0.9287
Round 1.952057013281503: Acc=0.9521, F1=0.9287

=== FEDADAM ===
Round 1.952057013281503: Acc=0.9521, F1=0.9287
Round 1.952057013281503: Acc=0.9521, F1=0.9287
Round 1.952057013281503: Acc=0.9521, F1=0.9287
Round 1.952057013281503: Acc=0.9521, F1=0.9287
Round 1.952057013281503: Acc=0.9521, F1=0.9287

FINAL RESULTS:
          Accuracy  Precision    Recall        F1
fedavg   0.952057   0.906413  0.952057  0.928674
fedprox  0.952057   0.906413  0.952057  0.928674
fedadam  0.952057   0.906413  0.952057  0.928674
